# Exploración inicial: NYC Taxi Trip Duration

Exploración rápida en Python. **No se modifica el dataset**, solo se lee y se describe.
La versión entregable de la exploración inicial se hará en Scala.

**Contenido**
1. Carga de datos
2. Tamaño, tipos y nulos
3. Estadísticas básicas y rango temporal
4. Clase (short / medium / long)
5. Variables categóricas y discretas
6. Duplicados e inconsistencias
7. Gráficas

In [43]:
## 1. Carga de datos

In [44]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PATH = "train.csv"
df = pd.read_csv(PATH, parse_dates=["pickup_datetime", "dropoff_datetime"])
df.head()

,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,N,455
1,id2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,N,663
2,id3858529,2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.979027,40.763939,-74.005333,40.710087,N,2124
3,id3504673,2,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.010040,40.719971,-74.012268,40.706718,N,429
4,id2181028,2,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.973053,40.793209,-73.972923,40.782520,N,435


## 2. Tamaño, tipos y nulos

In [45]:
print("Filas y columnas:", df.shape)
print("\nTipos de dato:")
print(df.dtypes)
print("\nNulos por columna:")
print(df.isnull().sum())

Filas y columnas: (1458644, 11)

Tipos de dato:
id                               str
vendor_id                      int64
pickup_datetime       datetime64[us]
dropoff_datetime      datetime64[us]
passenger_count                int64
pickup_longitude             float64
pickup_latitude              float64
dropoff_longitude            float64
dropoff_latitude             float64
store_and_fwd_flag               str
trip_duration                  int64
dtype: object

Nulos por columna:
id                    0
vendor_id             0
pickup_datetime       0
dropoff_datetime      0
passenger_count       0
pickup_longitude      0
pickup_latitude       0
dropoff_longitude     0
dropoff_latitude      0
store_and_fwd_flag    0
trip_duration         0
dtype: int64


## 3. Estadísticas básicas y rango temporal

In [46]:
# Estadísticas de las columnas numéricas
display(df.describe().T)

# Rango temporal real de los viajes (no confundir con las fechas de la competición)
print("Primer viaje:", df["pickup_datetime"].min())
print("Último viaje:", df["pickup_datetime"].max())

,count,mean,min,25%,50%,75%,max,std
vendor_id,1458644.0,1.53495,1.0,1.0,2.0,2.0,2.0,0.498777
pickup_datetime,1458644,2016-04-01 10:10:24.940038,2016-01-01 00:00:17,2016-02-17 16:46:04.250000,2016-04-01 17:19:40,2016-05-15 03:56:08.750000,2016-06-30 23:59:39,NaN
dropoff_datetime,1458644,2016-04-01 10:26:24.432309,2016-01-01 00:03:31,2016-02-17 17:05:32.500000,2016-04-01 17:35:12,2016-05-15 04:10:51.750000,2016-07-01 23:02:03,NaN
passenger_count,1458644.0,1.66453,0.0,1.0,1.0,2.0,9.0,1.314242
pickup_longitude,1458644.0,-73.973486,-121.933342,-73.991867,-73.981743,-73.967331,-61.335529,0.070902
pickup_latitude,1458644.0,40.750921,34.359695,40.737347,40.754101,40.76836,51.881084,0.032881
dropoff_longitude,1458644.0,-73.973416,-121.933304,-73.991325,-73.979752,-73.963013,-61.335529,0.070643
dropoff_latitude,1458644.0,40.7518,32.181141,40.735885,40.754524,40.76981,43.921028,0.035891
trip_duration,1458644.0,959.492273,1.0,397.0,662.0,1075.0,3526282.0,5237.431724


Primer viaje: 2016-01-01 00:00:17
Último viaje: 2016-06-30 23:59:39


## 4. Clase: short / medium / long

Se construye a partir de `trip_duration` (segundos):
- **short**: ≤ 10 min
- **medium**: entre 10 y 25 min (el 25 incluido)
- **long**: > 25 min

Es una variable temporal para explorar; el dataset original no se toca.

In [47]:
# Duración en minutos
columna_clase_min = df["trip_duration"] / 60

# np.select equivale a un if / elif / else vectorizado:
# toma la primera condición que se cumple; si ninguna, usa default
columna_clase_transformada = np.select(
    [columna_clase_min <= 10, columna_clase_min <= 25],
    ["short", "medium"],
    default="long"
)

# Convertimos el array de NumPy en una Series de pandas para poder usar value_counts()
orden = ["short", "medium", "long"]
cantidad_por_clase = pd.Series(columna_clase_transformada).value_counts().reindex(orden)
porcentaje_por_clase = (cantidad_por_clase / cantidad_por_clase.sum() * 100).round(2)

print("En total hay", len(cantidad_por_clase), "clases\n")
print("Recuento:")
print(cantidad_por_clase)
print("\nPorcentaje (%):")
print(porcentaje_por_clase)

En total hay 3 clases

Recuento:
short     652913
medium    625309
long      180422
Name: count, dtype: int64

Porcentaje (%):
short     44.76
medium    42.87
long      12.37
Name: count, dtype: float64


## 5. Variables categóricas y discretas

## 6. Duplicados e inconsistencias

Aquí se detectan los problemas que habrá que tratar en la limpieza.

## 7. Gráficas